[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powerzoojax/PowerZooPy/blob/main/docs/en/examples/notebooks/nb04_task_opf.ipynb)

# NB04 — Task 1: OPF

> **Prerequisites**: [NB03 — Obs / Action / Reward](./nb03_obs_action_reward.ipynb)  
> **Time**: ~15 minutes

## What You'll Learn

1. What "economic dispatch" means for an RL agent (no power systems knowledge required)
2. Full obs / action / reward structure for Task 1
3. How the train / val / test data splits map to real time series
4. Why the oracle score is the upper bound, and what "normalized score = 1" means

## Background: Economic Dispatch as a MARL Problem

**In power systems terms**: 5 generators must produce exactly enough power to meet
the total demand, while keeping current on each transmission line within safe limits.
Each generator has a different cost per MW — the goal is to produce cheaply.

**In RL terms**: 5 agents share a cooperative reward. Each agent's action is a
"preference score" — how much of the total load it wants to handle. The
environment converts these scores into actual MW via softmax normalization, then
runs a power flow simulation to check if the grid is safe.

```
Agent i  →  score_i ∈ [0, 1]
                ↓ softmax
        p_i = (score_i / Σ scores) × total_demand  [MW]
                ↓ DC power flow
        line_flows, voltages, cost  →  reward
```

**Why this is interesting for ML**: the optimal solution requires coordination —
each agent must "know" the other agents' costs and capacity limits to allocate
efficiently. Pure independent learning struggles; credit assignment is non-trivial.

## Setup

In [ ]:
import powerzoo
print(f"PowerZoo version: {powerzoo.__version__}")

In [ ]:
from powerzoo.tasks import make_task_env
from powerzoo.wrappers import GymnasiumWrapper
from powerzoo.benchmarks.policies import RandomPolicy, RuleBasedPolicy
from powerzoo.benchmarks import evaluate

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Task Configuration

In [ ]:
from powerzoo.tasks.simple.task1_marl_opf import MARLOPFTask

task = MARLOPFTask(split='train')

print("=== Task 1: marl_opf ===")
print(f"Name:        {task.name}")
print(f"Description: {task.description}")
print(f"Difficulty:  {task.difficulty}")
print(f"\nData splits:")
for split, (start, end) in task.SPLIT_DATES.items():
    print(f"  {split:6s}: {start} → {end}")
print(f"\nEvaluation protocol:")
for k, v in task.eval_protocol.items():
    print(f"  {k}: {v}")

## 2. Data Splits on the Timeline

All three splits are driven by bundled real system demand time series (half-hourly, 2016–2025).
The test split (2024) was not seen during training — it represents genuine out-of-sample generalization.

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 1.8))

splits = {
    'train': (2016, 2023, '#4CAF50', 'Train (2016–2022)'),
    'val':   (2023, 2024, '#FF9800', 'Val (2023)'),
    'test':  (2024, 2025, '#E91E63', 'Test (2024) ← benchmark'),
}

for split, (start, end, color, label) in splits.items():
    ax.barh(0, end - start, left=start, height=0.4, color=color, alpha=0.85,
            label=label)
    ax.text((start + end) / 2, 0, f'{end - start}y', ha='center', va='center',
            fontweight='bold', color='white', fontsize=10)

ax.set_xlim(2014, 2026)
ax.set_yticks([])
ax.set_xlabel('Year')
ax.set_title('Task 1 data splits — half-hourly demand traces', fontsize=11)
ax.legend(loc='upper left', ncol=3, framealpha=0.9)

# Add data availability note
ax.annotate('Trace window starts', xy=(2016, 0.25), xytext=(2014.5, 0.25),
            fontsize=8, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

plt.tight_layout()
plt.show()
print("\nNote: all three tasks share identical split boundaries.")

## 3. Environment Details: Agents, Obs, Action, Reward

In [ ]:
env = make_task_env("marl_opf", split="train")
gym_env = GymnasiumWrapper(env)

obs, info = gym_env.reset(seed=42)

print("=== Environment specs ===")
print(f"Agents:           {env.agents}")
print(f"Episode length:   48 steps (1 day @ 30-min intervals)")
print(f"\nGym obs shape:    {obs.shape}")
print(f"Gym act shape:    {gym_env.action_space.shape}")
print(f"Action range:     [{gym_env.action_space.low[0]:.1f}, {gym_env.action_space.high[0]:.1f}]")

print("\n=== Obs interpretation ===")
print("  Global: total system load (normalized), line flows, time-of-day (sin/cos)")
print("  Local:  unit index, p_min, p_max, cost coefficients")
print("  → ML analogy: global state = shared environment; local = agent-specific features")

## 4. Running Episodes: Random vs Rule-Based Policy

In [ ]:
def run_episode(env, policy, seed=42):
    """Run one episode; return step rewards and cumulative reward."""
    obs, info = env.reset(seed=seed)
    if hasattr(policy, 'reset'):
        policy.reset()
    step_rewards = []
    while True:
        action = policy.act(obs, info)
        obs, reward, terminated, truncated, info = env.step(action)
        step_rewards.append(reward)
        if terminated or truncated:
            break
    return step_rewards, sum(step_rewards)

gym_env = GymnasiumWrapper(make_task_env("marl_opf", split="train"))

# Random policy
random_policy = RandomPolicy(gym_env.action_space, seed=0)
random_rewards, random_total = run_episode(gym_env, random_policy)

# Rule-based policy (equal allocation — "proportional dispatch")
rule_policy = RuleBasedPolicy(gym_env.action_space)
rule_rewards, rule_total = run_episode(gym_env, rule_policy)

print(f"Random policy total reward:     {random_total:.2f}")
print(f"Rule-based policy total reward: {rule_total:.2f}")
print(f"Difference: {rule_total - random_total:+.2f}")

In [ ]:
# Compare step-by-step
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

steps = range(len(random_rewards))
ax1.plot(steps, random_rewards, label='Random', color='tomato', alpha=0.8)
ax1.plot(steps, rule_rewards, label='Rule-based', color='steelblue', alpha=0.8)
ax1.axhline(0, color='gray', linewidth=0.7, linestyle='--')
ax1.set_ylabel('Step reward')
ax1.legend()
ax1.set_title('Task 1 (OPF): step reward comparison')

# Cumulative reward
ax2.plot(steps, np.cumsum(random_rewards), label='Random', color='tomato')
ax2.plot(steps, np.cumsum(rule_rewards), label='Rule-based', color='steelblue')
ax2.set_xlabel('Timestep (30-min interval)')
ax2.set_ylabel('Cumulative reward')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Oracle Upper Bound

The oracle policy solves the OPF exactly using a mathematical solver (Gurobi/GLPK).
Its mean reward defines `normalized_score = 1.0` — the best achievable upper bound.

The oracle is accessed via `OraclePolicy`:

In [ ]:
from powerzoo.benchmarks.policies import OraclePolicy

# OraclePolicy wraps the environment's built-in OPF solver
oracle_env = GymnasiumWrapper(make_task_env("marl_opf", split="train"))
oracle_policy = OraclePolicy(oracle_env)

oracle_rewards, oracle_total = run_episode(oracle_env, oracle_policy)
print(f"Oracle policy total reward:     {oracle_total:.2f}")
print(f"Rule-based policy total reward: {rule_total:.2f}")
print(f"Random policy total reward:     {random_total:.2f}")
print()
print("Manual normalized score (rule-based):")
ns = (rule_total - random_total) / (oracle_total - random_total + 1e-8)
print(f"  (rule - random) / (oracle - random) = {ns:.4f}")

In [ ]:
# Visual comparison: three policies on the same episode
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(np.cumsum(random_rewards), color='tomato', linewidth=2, label=f'Random (score=0.0)')
ax.plot(np.cumsum(rule_rewards), color='steelblue', linewidth=2,
        label=f'Rule-based (score={ns:.2f})')
ax.plot(np.cumsum(oracle_rewards), color='#4CAF50', linewidth=2, linestyle='--',
        label='Oracle (score=1.0)')

ax.set_xlabel('Timestep')
ax.set_ylabel('Cumulative reward')
ax.set_title('Task 1 — Cumulative reward: Random vs Rule-based vs Oracle')
ax.legend()
plt.tight_layout()
plt.show()

print("\nGoal: train an RL agent whose curve approaches the Oracle line.")

## 6. Variants

Task 1 also comes in larger variants for harder benchmarks:

In [ ]:
from powerzoo.tasks import list_tasks

opf_tasks = [t for t in list_tasks() if 'opf' in t]
print("OPF task variants:")
for t in opf_tasks:
    e = make_task_env(t, split='train')
    g = GymnasiumWrapper(e)
    obs, _ = g.reset(seed=0)
    print(f"  {t:30s}  agents={len(e.agents)}  obs_dim={obs.shape[0]}")

---

## Summary

| Aspect | Task 1 (marl_opf) |
|---|---|
| Grid | IEEE 5-bus transmission, DC power flow |
| Agents | 5 generators; each controls output allocation score ∈ [0, 1] |
| Episode | 48 steps = 1 day @ 30-min intervals |
| Reward | −generation cost − line overload penalty |
| Oracle | Exact DC-OPF (Gurobi/GLPK); normalized score upper bound = 1.0 |
| Variants | `marl_opf_7d` (7-day), `marl_opf_118` (118-bus, harder) |
| Key challenge | Cooperative cost minimization with coupled safety constraints |

## Next

→ [NB05 — Task 2: Battery Storage Arbitrage (What's Different)](./nb05_task_der.ipynb)